In [3]:
from app.database.connection import check_database_connection
from app.database.query_executor import fetch_dataframe

print("Connection test:", check_database_connection())

views = fetch_dataframe("""
    SELECT table_name
    FROM information_schema.views
    WHERE table_schema = 'olist'
    ORDER BY table_name;
""")

print(views)

Connection test: 1
             table_name
0      vw_category_kpis
1      vw_customer_kpis
2       vw_customer_rfm
3  vw_customer_segments
4       vw_monthly_kpis
5    vw_order_analytics
6  vw_payment_analytics
7       vw_product_kpis
8   vw_review_analytics
9        vw_seller_kpis


In [4]:

from datetime import date

from app.database.connection import check_database_connection

from app.analytics.marketplace import (
    get_marketplace_kpis,
    get_monthly_marketplace_trends,
    get_cancellation_trend
)

# 1. Database connection
print("Database connection:", check_database_connection())

# 2. Overall marketplace KPIs
print("\n========== MARKETPLACE KPIs ==========")

kpis = get_marketplace_kpis()

for metric, value in kpis.items():
    print(f"{metric}: {value}")

# 3. Monthly trends
print("\n========== MONTHLY TRENDS ==========")

monthly = get_monthly_marketplace_trends()

print(monthly.to_string(index=False))

# 4. Cancellation trends
print("\n========== CANCELLATION TRENDS ==========")

cancellations = get_cancellation_trend()

print(cancellations.to_string(index=False))

# 5. Test date filtering
print("\n========== DATE FILTER TEST ==========")

filtered = get_marketplace_kpis(
    start_date=date(2018, 1, 1),
    end_date=date(2018, 3, 31)
)

print(filtered)


Database connection: 1

========== MARKETPLACE KPIs ==========
total_orders: 99441.0
delivered_orders: 96478.0
canceled_orders: 625.0
unique_customers: 96096.0
ordered_product_value: 13591643.7
delivered_gmv: 13221498.11
delivered_aov: 137.04
eligible_delivered_orders: 96470.0
late_orders: 6534.0
late_delivery_rate: 6.77
avg_delivery_days: 12.56
cancellation_rate: 0.63

========== MONTHLY TRENDS ==========
     month  total_orders  delivered_orders  canceled_orders  unique_customers  ordered_product_value  delivered_gmv  delivered_aov  eligible_delivered_orders  late_orders  late_delivery_rate  avg_delivery_days  cancellation_rate
2016-09-01             4                 1                2                 4                 267.36         134.97         134.97                          1            1              100.00              54.81              50.00
2016-10-01           324               265               24               321               49507.66       40325.11         152.17  

In [5]:
from sqlalchemy import text
from app.database.connection import engine

VIEWS = [
    "vw_customer_kpis",
    "vw_customer_rfm",
    "vw_customer_segments"
]

def inspect_customer_views():
    query = text("""
        SELECT
            table_name,
            column_name,
            data_type,
            ordinal_position
        FROM information_schema.columns
        WHERE table_schema = 'olist'
          AND table_name = ANY(:view_names)
        ORDER BY table_name, ordinal_position;
    """)

    with engine.connect() as connection:
        result = connection.execute(
            query,
            {"view_names": VIEWS}
        )

        rows = result.fetchall()

    for view in VIEWS:
        print(f"\n{'=' * 65}")
        print(f"VIEW: olist.{view}")
        print("=" * 65)

        view_columns = [
            row for row in rows if row[0] == view
        ]

        if not view_columns:
            print("No columns found. Check that the view exists.")
            continue

        for row in view_columns:
            print(f"{row[1]:35} {row[2]}")


if __name__ == "__main__":
    inspect_customer_views()


VIEW: olist.vw_customer_kpis
customer_unique_id                  text
total_orders                        bigint
delivered_orders                    bigint
canceled_orders                     bigint
total_product_value                 numeric
delivered_product_value             numeric
total_freight_value                 numeric
total_recorded_payment              numeric
first_order_date                    timestamp without time zone
last_order_date                     timestamp without time zone
delivered_aov                       numeric
customer_lifespan_days              integer
recency_days                        integer
frequency_segment                   text

VIEW: olist.vw_customer_rfm
customer_unique_id                  text
last_purchase_date                  date
reference_date                      date
recency                             integer
frequency                           bigint
monetary                            numeric
r_score                             inte

In [7]:

from app.database.connection import check_database_connection

from app.analytics.customers import (
    get_customer_overview,
    get_customer_frequency_distribution,
    get_customer_segments,
    get_rfm_distribution
)


def main():

    print("\nDATABASE CONNECTION")
    print(check_database_connection())

    # 1. Customer overview
    print("\n" + "=" * 70)
    print("1. CUSTOMER OVERVIEW")
    print("=" * 70)

    overview = get_customer_overview()

    for metric, value in overview.items():
        print(f"{metric:45}: {value}")

    # 2. Frequency distribution
    print("\n" + "=" * 70)
    print("2. CUSTOMER FREQUENCY DISTRIBUTION")
    print("=" * 70)

    frequency = get_customer_frequency_distribution()

    print(frequency.to_string(index=False))

    # 3. Business customer segments
    print("\n" + "=" * 70)
    print("3. CUSTOMER BUSINESS SEGMENTS")
    print("=" * 70)

    segments = get_customer_segments()

    print(segments.to_string(index=False))

    # 4. RFM distribution
    print("\n" + "=" * 70)
    print("4. RFM DISTRIBUTION")
    print("=" * 70)

    rfm = get_rfm_distribution()

    print(rfm.to_string(index=False))


if __name__ == "__main__":
    main()



DATABASE CONNECTION
1

1. CUSTOMER OVERVIEW
total_customers                              : 96096.0
customers_with_delivered_orders              : 93358.0
repeat_customers                             : 2801.0
total_orders                                 : 99441.0
total_delivered_orders                       : 96478.0
total_canceled_orders                        : 625.0
delivered_gmv                                : 13221498.11
avg_delivered_spend_per_customer             : 141.62
avg_customer_aov                             : 137.51
avg_delivered_orders_per_active_customer     : 1.03
repeat_customer_rate                         : 3.0

2. CUSTOMER FREQUENCY DISTRIBUTION
frequency_segment  total_customers  customer_percentage  avg_total_orders  avg_delivered_spend
         One-time            93099                96.88              1.00               133.85
       Occasional             2948                 3.07              2.07               247.14
         Frequent               44   

In [8]:

from app.database.connection import check_database_connection

from app.analytics.products import (
    get_product_overview,
    get_top_products,
    get_category_performance,
    get_category_contribution,
    get_product_price_distribution,
    get_product_performance_summary,
)


def print_section(title):
    print("\n" + "=" * 75)
    print(title)
    print("=" * 75)


def main():
    print_section("DATABASE CONNECTION")
    print(check_database_connection())

    print_section("1. PRODUCT OVERVIEW")
    overview = get_product_overview()
    for metric, value in overview.items():
        print(f"{metric:40}: {value}")

    print_section("2. TOP 10 PRODUCTS BY DELIVERED GMV")
    print(
        get_top_products(
            metric="delivered_gmv",
            limit=10
        ).to_string(index=False)
    )

    print_section("3. TOP 10 PRODUCTS BY DELIVERED ITEM VOLUME")
    print(
        get_top_products(
            metric="delivered_items",
            limit=10
        ).to_string(index=False)
    )

    print_section("4. CATEGORY PERFORMANCE")
    print(get_category_performance().to_string(index=False))

    print_section("5. CATEGORY GMV CONTRIBUTION")
    print(get_category_contribution().to_string(index=False))

    print_section("6. PRODUCT PRICE DISTRIBUTION")
    print(get_product_price_distribution().to_string(index=False))

    print_section("7. PRODUCT PERFORMANCE SUMMARY")
    summary = get_product_performance_summary()
    for metric, value in summary.items():
        print(f"{metric:40}: {value}")


if __name__ == "__main__":
    main()



DATABASE CONNECTION
1

1. PRODUCT OVERVIEW
total_products                          : 32951.0
products_with_sales                     : 32951.0
products_with_deliveries                : 32216.0
total_items_sold                        : 112650.0
delivered_items                         : 110197.0
canceled_items                          : 542.0
delivered_gmv                           : 13221498.11
total_product_value                     : 13591643.7
total_freight_value                     : 2251909.54
avg_product_selling_price               : 145.3
avg_delivered_item_price                : 119.98
product_order_combinations              : 102425.0

2. TOP 10 PRODUCTS BY DELIVERED GMV
                      product_id      product_category  total_items_sold  total_orders  unique_customers  unique_sellers  delivered_items  delivered_orders  delivered_gmv  avg_selling_price  canceled_items
bb50f2e236e5eea0100680137654686c         health_beauty               195           187               187 

In [1]:

from app.database.connection import check_database_connection

from app.analytics.sellers import (
    get_seller_overview,
    get_top_sellers,
    get_seller_delivery_performance,
    get_seller_contribution,
    get_seller_performance_summary,
)


def main():

    print("\n" + "=" * 75)
    print("DATABASE CONNECTION")
    print("=" * 75)

    print(check_database_connection())

    # print("\n" + "=" * 75)
    # print("1. SELLER OVERVIEW")
    # print("=" * 75)

    # print(get_seller_overview().T)

    # print("\n" + "=" * 75)
    # print("2. TOP 10 SELLERS BY DELIVERED GMV")
    # print("=" * 75)

    # print(get_top_sellers(metric="delivered_gmv", limit=10))

    # print("\n" + "=" * 75)
    # print("3. TOP 10 SELLERS BY DELIVERED ORDERS")
    # print("=" * 75)

    # print(get_top_sellers(metric="delivered_orders", limit=10))

    # print("\n" + "=" * 75)
    # print("4. SELLER DELIVERY PERFORMANCE")
    # print("=" * 75)

    # print(get_seller_delivery_performance(limit=20))

    # print("\n" + "=" * 75)
    # print("5. SELLER GMV CONTRIBUTION")
    # print("=" * 75)

    # print(get_seller_contribution(limit=20))

    print("\n" + "=" * 75)
    print("6. SELLER PERFORMANCE SUMMARY")
    print("=" * 75)

    print(get_seller_performance_summary().T)


if __name__ == "__main__":
    main()



DATABASE CONNECTION
1

6. SELLER PERFORMANCE SUMMARY
                                                                    0
total_sellers                                                    3095
total_items_sold                                             112650.0
delivered_items                                              110197.0
seller_order_associations                                    100010.0
delivered_seller_order_associations                           97819.0
delivered_gmv                                             13221498.11
avg_seller_freight_value                                        23.38
seller_customer_relationships                                100010.0
eligible_seller_order_associations                            97811.0
late_seller_order_associations                                 6547.0
late_delivery_rate                                               6.69
top_seller_id                        4869f7a5dfa277a7dca6462dcf3b52b2
top_seller_gmv                      

In [2]:

from app.database.connection import check_database_connection

from app.analytics.payments import (
    get_payment_overview,
    get_payment_method_distribution,
    get_payment_installment_analysis,
    get_payment_value_distribution,
    get_payment_anomalies,
    get_payment_analytics_summary,
)


def main():
    print("\nChecking database connection...")
    if not check_database_connection():
        raise RuntimeError("Database connection failed")

    print("\n" + "=" * 75)
    print("1. PAYMENT OVERVIEW")
    print("=" * 75)
    print(get_payment_overview().to_string(index=False))

    print("\n" + "=" * 75)
    print("2. PAYMENT METHOD DISTRIBUTION")
    print("=" * 75)
    print(get_payment_method_distribution().to_string(index=False))

    print("\n" + "=" * 75)
    print("3. PAYMENT INSTALLMENT ANALYSIS")
    print("=" * 75)
    print(get_payment_installment_analysis().to_string(index=False))

    print("\n" + "=" * 75)
    print("4. PAYMENT VALUE DISTRIBUTION")
    print("=" * 75)
    print(get_payment_value_distribution().to_string(index=False))

    print("\n" + "=" * 75)
    print("5. PAYMENT ANOMALIES")
    print("=" * 75)
    print(get_payment_anomalies().to_string(index=False))

    print("\n" + "=" * 75)
    print("6. PAYMENT ANALYTICS SUMMARY")
    print("=" * 75)
    print(get_payment_analytics_summary().to_string(index=False))


if __name__ == "__main__":
    main()



Checking database connection...

1. PAYMENT OVERVIEW
 total_payment_records  unique_orders  unique_customers  delivered_payment_records  delivered_orders  canceled_orders  total_recorded_payment_value  delivered_order_recorded_payment_value  avg_payment_record_value  avg_recorded_payment_per_order
                103886          99440             96095                     100756             96477              625                   16008872.12                             15422461.77                     154.1                          160.99

2. PAYMENT METHOD DISTRIBUTION
payment_type  payment_records  unique_orders  unique_customers  total_payment_value  record_percentage  payment_value_percentage
 credit_card            76795          76505             74092          12542084.19              73.92                     78.34
      boleto            19784          19784             19290           2869361.27              19.04                     17.92
     voucher             5775      

In [3]:

from app.database.connection import check_database_connection

from app.analytics.reviews import (
    get_review_overview,
    get_review_score_distribution,
    get_review_sentiment_analysis,
    get_review_delivery_relationship,
    get_review_monthly_trend,
    get_review_analytics_summary,
)


def main():
    print("\nChecking database connection...")

    if not check_database_connection():
        raise RuntimeError("Database connection failed")

    print("\n" + "=" * 75)
    print("1. REVIEW OVERVIEW")
    print("=" * 75)
    print(get_review_overview().to_string(index=False))

    print("\n" + "=" * 75)
    print("2. REVIEW SCORE DISTRIBUTION")
    print("=" * 75)
    print(get_review_score_distribution().to_string(index=False))

    print("\n" + "=" * 75)
    print("3. REVIEW SENTIMENT ANALYSIS")
    print("=" * 75)
    print(get_review_sentiment_analysis().to_string(index=False))

    print("\n" + "=" * 75)
    print("4. REVIEW DELIVERY RELATIONSHIP")
    print("=" * 75)
    print(get_review_delivery_relationship().to_string(index=False))

    print("\n" + "=" * 75)
    print("5. MONTHLY REVIEW TREND")
    print("=" * 75)
    print(get_review_monthly_trend().to_string(index=False))

    print("\n" + "=" * 75)
    print("6. REVIEW ANALYTICS SUMMARY")
    print("=" * 75)
    print(get_review_analytics_summary().to_string(index=False))


if __name__ == "__main__":
    main()



Checking database connection...

1. REVIEW OVERVIEW
 total_review_records  unique_reviews  reviewed_orders  unique_customers  duplicate_review_id_records  missing_review_scores  invalid_review_scores  average_review_score  negative_reviews  negative_review_percentage  reviews_with_comments  reviews_with_titles
                99224           98410            98673             95380                          814                      0                      0                  4.09             14575                       14.69                  40968                11566

2. REVIEW SCORE DISTRIBUTION
 review_score  review_records  unique_orders  review_percentage
            1           11424          11393              11.51
            2            3151           3148               3.18
            3            8179           8160               8.24
            4           19142          19098              19.29
            5           57328          57076              57.78

3. REVIEW SE